# STM32 ADC $R_{ain}$ Calcuations


$$ R_{ain} (max) = \frac{k - 0.5}{f_{ADC} \cdot C_{ADC} \cdot ln(2^{N-2})} - R_{ADC}$$


"The formula above is used to determine the maximum external impedance <br>
allowed for an error below 1/4 of LSB. N = 12 (from 12-bit resolution) and k is the number of <br>
sampling periods defined in the ADC_SMPR1 register." From STM32F466 datasheet

In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
# Goal is Rain >= R_actual
R_actual = 10e3 # impedance of circuit going to ADC.

In [9]:
# STM32F446 Datasheet
Radc = 6e3      # 6k ohms (FIXED)
Cadc = 7e-12    # 4-7 pF (use 7pF for worst case)
fadc = 22.5e6   # Frequency of ADC, this depends which timer your using. (APB1 or APB2).
N = 12          # ADC Bit Resolution, depends on STM32 Chip. (FIXED)
ks = np.array([3, 15, 28, 56, 84, 112, 144, 480]) # (FIXED)

In [10]:
def Rain_f(k):
    R = (k - 0.5) / (fadc * Cadc * np.log(pow(2, N + 2))) - Radc
    return float(R)

In [11]:
for k in ks:
    passes = Rain_f(k) >= R_actual
    print(f"{passes}: k = {k}, Rain = {Rain_f(k):.0f}")
    if passes:
        print(f"Smallest valid k = {k}")
        print(f"Valid k's = {[float(x) for x in ks if x >= k]}")
        break

False: k = 3, Rain = -4364
False: k = 15, Rain = 3487
True: k = 28, Rain = 11993
Smallest valid k = 28
Valid k's = [28.0, 56.0, 84.0, 112.0, 144.0, 480.0]
